# Choose the threshold on validation, then check test once

The business learner sets the costs. The engineering learner writes the code that applies them.

A false negative is a churner who got no voucher. Default cost: 500 THB.

A false positive is a voucher sent to someone who would have ordered anyway. Default cost: 100 THB.

If your team changes these costs, edit them in the cost cell before running the test cell. The threshold search uses only validation rows. Run the test cell once. Do not change the model after seeing the test result.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    if (candidate / "schema.py").is_file() and (candidate / "app.py").is_file():
        root = candidate
        break
    nested = candidate / "01-supervised-learning"
    if (nested / "schema.py").is_file() and (nested / "app.py").is_file():
        root = nested
        break
if root is None:
    raise FileNotFoundError(
        "Start Jupyter from the repo folder, from 01-supervised-learning, or from its notebooks folder."
    )
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
ROOT = root
print("cwd", here)
print("module", ROOT)


In [ ]:
import pandas as pd

customers = pd.read_csv(ROOT / "data" / "fresh_and_good_customers.csv")
print(customers.shape)
customers.head()


In [ ]:
from frame import features_and_target
from split import split_customers

train, validation, test = split_customers(customers)
print("train", len(train), "validation", len(validation), "test", len(test))
X_train, y_train = features_and_target(train)
X_val, y_val = features_and_target(validation)
X_test, y_test = features_and_target(test)
print("churned is a label, not a feature:", "churned" not in X_train.columns)


In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score

from pipelines import make_forest_pipeline, make_logistic_pipeline, tune_forest
from schema import DEFAULT_COST_FN_THB, DEFAULT_COST_FP_THB
from threshold import choose_threshold, labels_at, total_cost

logistic = make_logistic_pipeline()
logistic.fit(X_train, y_train)
forest = make_forest_pipeline()
forest.fit(X_train, y_train)
search = tune_forest(X_train, y_train, n_iter=8)
tuned = search.best_estimator_


def validation_auc(model):
    return float(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))


scores = {
    "logistic": validation_auc(logistic),
    "forest": validation_auc(forest),
    "tuned_forest": validation_auc(tuned),
}
winner_name = max(scores, key=scores.get)
winner = {"logistic": logistic, "forest": forest, "tuned_forest": tuned}[winner_name]
print({name: round(value, 3) for name, value in scores.items()})
print("winner", winner_name)


Read the validation cost before checking test rows. A low threshold sends more vouchers when missed churners cost more than wasted vouchers. To reduce the list, raise `COST_FP` and rerun the cost cell before running the test cell.


In [ ]:
COST_FN = DEFAULT_COST_FN_THB
COST_FP = DEFAULT_COST_FP_THB
val_proba = winner.predict_proba(X_val)[:, 1]
threshold = choose_threshold(y_val, val_proba, COST_FN, COST_FP)
val_pred = labels_at(val_proba, threshold)
print("threshold", threshold)
print("validation", total_cost(y_val, val_pred, COST_FN, COST_FP))
print(confusion_matrix(y_val, val_pred))
print("Top row is actual stay. Bottom row is actual churn.")
print("Left column is predicted stay. Right column is predicted churn.")


### Exercise. Test set, once

Assign precision, recall, and ROC-AUC on the test rows.

`precision_score(y_test, test_pred, zero_division=0)`

`recall_score(y_test, test_pred, zero_division=0)`

`roc_auc_score(y_test, test_proba)` uses probabilities, not the yes or no labels.


In [ ]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score

test_proba = winner.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= threshold).astype(int)
test_precision = None
test_recall = None
test_auc = None


In [ ]:
if None in (test_precision, test_recall, test_auc):
    raise NotImplementedError("Assign test precision, recall, and ROC-AUC. Run the test cell once.")
print("test precision", round(float(test_precision), 3))
print("test recall", round(float(test_recall), 3))
print("test ROC-AUC", round(float(test_auc), 3))
print(total_cost(y_test, test_pred, COST_FN, COST_FP))


In [ ]:
from persist import save_bundle

destination = ROOT / "models" / "churn_bundle.joblib"
save_bundle(destination, winner, threshold, COST_FN, COST_FP)
print("saved", destination)
print("Load only this file or one your instructor hands you.")
print("A joblib file can run code when it is loaded.")


## Optional engineer stretch

Skip this in the timed demo.

Fit `GradientBoostingClassifier(n_estimators=30, max_depth=2, random_state=42)` after the same prep used in `pipelines.py`. Fit on the train rows only. Print validation ROC-AUC. For the forest, print `feature_importances_` from the model step and line them up with `prep.get_feature_names_out()`.

Do not overwrite `models/churn_bundle.joblib` unless the team repeats the validation threshold step for the new model.

In `app.py`, the expander under the form is the input-validation stretch.


## Checkpoint

The app reads `models/churn_bundle.joblib` and `models/spend_bundle.joblib`. Next is the short spend notebook, then the form.
